# Subset an `.h5ad` by annotation column

Pick which `global.cluster4` (or any obs column) labels to keep, then save a new `.h5ad`.

**Workflow:** run cell 1 (load) → cell 2 (see available labels) → edit cell 3 (config) → run cell 4 (subset + save).

**Note on embeddings:** `X_pca`, `X_harmony`, `X_umap` are *sliced*, not recomputed. The subset's UMAP still reflects the full-dataset manifold. Recompute downstream if you need a subset-specific embedding.

In [1]:
import anndata as ad

INPUT_PATH = "data/hnc_myeloid_2021.h5ad"
adata = ad.read_h5ad(INPUT_PATH)
print(adata)

AnnData object with n_obs × n_vars = 26444 × 23630
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'seurat_clusters', 'RNA_snn_res.0.5', 'RNA_snn_res.0.6', 'RNA_snn_res.0.7', 'RNA_snn_res.0.8', 'RNA_snn_res.0.9', 'RNA_snn_res.1', 'RNA_snn_res.1.1', 'RNA_snn_res.1.2', 'RNA_snn_res.1.3', 'RNA_snn_res.1.4', 'RNA_snn_res.1.5', 'RNA_snn_res.1.6', 'RNA_snn_res.1.7', 'RNA_snn_res.1.8', 'RNA_snn_res.1.9', 'RNA_snn_res.2', 'tissue', 'is_HD', 'global.cluster', 'global.cluster2', 'hpv_status', 'RNA_snn_res.2.1', 'RNA_snn_res.2.2', 'RNA_snn_res.2.3', 'RNA_snn_res.2.4', 'RNA_snn_res.2.5', 'RNA_snn_res.2.6', 'RNA_snn_res.2.7', 'RNA_snn_res.2.8', 'RNA_snn_res.2.9', 'RNA_snn_res.3', 'tissue_hpv', 'global.cluster3', 'global.cluster4'
    obsm: 'X_harmony', 'X_pca', 'X_umap'


## See available labels
Copy the exact strings you want from the printout below — names are case- and underscore-sensitive.

In [2]:
COLUMN = "global.cluster4"  # change to any obs column

counts = adata.obs[COLUMN].astype(str).value_counts()
print(f"{COLUMN} — {len(counts)} labels, {adata.n_obs} cells total\n")
print(counts.to_string())

global.cluster4 — 17 labels, 26444 cells total

global.cluster4
Mono_CD14          7518
Mono_CD16          2503
Mac_IL1Bint        1842
Mono_CD14_IL1B     1801
Mono_CD14_THBS1    1783
Mono_Int           1684
cDC2_CD33          1319
Mac_CXCL9          1225
DC_pDC             1218
Mono_CD14_ID1      1154
Mono_TIL            988
cDC2_CD1C           966
Mac_IL1B            774
Mac_SPP1            573
mregDC_LAMP3        482
Mast                313
cDC1_CLEC9A         301


## Config — EDIT THIS CELL
Paste the labels you want to keep as a comma-separated string.

In [3]:
# ===== EDIT =====
KEEP = "Mast"           
OUTPUT_PATH = "data/hnc_myeloid_2021_mast.h5ad"
# ================

wanted = [x.strip() for x in KEEP.split(",") if x.strip()]
print("Will keep:", wanted)

Will keep: ['Mast']


## Validate, subset, and save
Errors loudly if any label is misspelled (so you never get a silent empty subset).

In [4]:
# Validate names against actual categories
available = set(adata.obs[COLUMN].astype(str).unique())
missing = [w for w in wanted if w not in available]
if missing:
    raise ValueError(
        f"Not found in '{COLUMN}': {missing}\n"
        f"Available: {sorted(available)}"
    )

# Subset (.copy() materializes the view so write is clean)
mask = adata.obs[COLUMN].astype(str).isin(wanted)
sub = adata[mask].copy()

# Drop now-unused categories from ALL categorical obs columns
for col in sub.obs.select_dtypes("category").columns:
    sub.obs[col] = sub.obs[col].cat.remove_unused_categories()

print(f"Kept {sub.n_obs} / {adata.n_obs} cells\n")
print(sub.obs[COLUMN].value_counts().to_string())

sub.write_h5ad(OUTPUT_PATH)
print(f"\nSaved → {OUTPUT_PATH}")

Kept 313 / 26444 cells

global.cluster4
Mast    313

Saved → data/hnc_myeloid_2021_mast.h5ad
